In [1]:
# orb_15min_retest_sp500_validation.py
# ORB (15m) with Retest/Continuation entries + Validation Suite
# - Scale out in thirds at 1R / 2R / 3R; BE after +1R; optional ATR trail (last third after TP2)
# - Daily features are shifted to avoid look-ahead bias
# - Validation tests print Win% per test + overall summary

import warnings
warnings.filterwarnings("ignore")

import time
from datetime import timedelta, time as dtime
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd
import requests
import yfinance as yf

# =============================
# USER PARAMETERS
# =============================

# Data (Yahoo intraday <60m => ~last 60 trading days)
INTERVAL = "15m"
PERIOD = "60d"
USE_PERIOD = True

# Session / timezone
TZ = "America/New_York"
REG_SESSION_START = "09:30"
REG_SESSION_END   = "16:00"

# ORB logic (quality + enough trades)
MAX_RETEST_MIN = 120
RETEST_CONFIRM_CLOSE = True
BREAKOUT_CUSHION_PCT = 0.0012    # 0.12% cushion beyond OR to confirm breakout/retest
MAX_OPPOSITE_WICK_FRAC = 0.30    # reject entry bar if opposite wick > 30% of bar range

# Multiple trades per day
MAX_TRADES_PER_SESSION = 2
ALLOW_BOTH_SIDES = True

# Scale-out in thirds (boosts avg R while keeping WR)
USE_SCALE_OUT = True
TARGETS_R = [1.0, 2.0, 3.0]      # 1R, 2R, 3R
SCALE_SPLIT = (1/3, 1/3, 1/3)
MOVE_STOP_TO_BE_AT_R = 1.0       # BE after TP1 (+1R)

# Continuation fallback if no retest
USE_CONTINUATION = True

# Time windows (early to avoid chop)
BREAKOUT_DEADLINE   = "10:45"
RETEST_DEADLINE     = "11:30"
CONTINUATION_CUTOFF = "11:30"

# Filters (balanced for more trades + WR)
USE_VWAP_FILTER   = True
USE_TREND_FILTER  = True          # 15m EMA trend filter
EMA_FAST = 20
EMA_SLOW = 50
# EMA slope helper (bps of price across last 5 bars)
MIN_EMA_SLOPE_FRAC = 0.0002       # ≈ 2 bps

USE_RSI_FILTER = True
RSI_LEN = 14
RSI_THRESH_LONG = 55.0
RSI_THRESH_SHORT = 45.0

USE_ADX_FILTER = True
ADX_LEN = 14
ADX_MIN = 20.0

USE_DAILY_TREND_FILTER = True
DAILY_EMA_FAST = 20
DAILY_EMA_SLOW = 50
DAILY_ATR_LEN  = 14

# Opening range width vs Daily ATR band
USE_OR_WIDTH_ATR_FILTER = True
OR_ATR_MIN = 0.15
OR_ATR_MAX = 1.20

# Opening 15m volume quantile gate
USE_OPENING_VOL_FILTER = True
OPENING_VOL_LOOKBACK = 60
OPENING_VOL_QUANTILE = 0.40

# Gap filters
USE_GAP_FILTER = True
GAP_MIN = 0.003                 # 0.3%
GAP_MAX = 0.04                  # 4.0%
ALIGN_WITH_GAP_DIR = False

USE_DOW_FILTER = False
ALLOWED_DOW = {1, 2, 3}         # Tue–Thu

# Optional index confirmation
USE_SPY_CONFIRM = True
SPY_TICKER = "SPY"

# Risk / costs
R_MULTIPLES = [2.0]             # unused when USE_SCALE_OUT=True
POSITION_SIZE_DOLLARS = 10_000
SLIPPAGE_BPS = 1.0
FEES_PER_TRADE = 0.00

# Trailing on last third after TP2
USE_ATR_TRAIL = True
ATR15_LEN = 14
ATR15_MULT = 2.5

# Batch / persistence
SLEEP_BETWEEN_TICKERS = 0.35
AUTOSAVE_EVERY = 25

# Output files
TRADES_CSV_ALL = "orb_trades_sp500.csv"
EQUITY_CSV_ALL = "orb_equity_sp500.csv"
SUMMARY_CSV    = "orb_summary_sp500.csv"

# ========== Helpers / core strategy bits ==========

def ensure_unique_columns(df: pd.DataFrame) -> pd.DataFrame:
    if getattr(df.columns, "duplicated", None) is not None and df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()].copy()
    return df

def sessionize(obj) -> pd.Series:
    idx = getattr(obj, "index", obj)
    return pd.to_datetime(pd.Index(idx).date)

def _to_clock(ts) -> dtime:
    return dtime(ts.hour, ts.minute)

def _clock_le(ts, hhmm: str) -> bool:
    h, m = map(int, hhmm.split(":"))
    return _to_clock(ts) <= dtime(h, m)

def wick_opposite_fraction(row: pd.Series, direction: str) -> float:
    h, l, o, c = float(row["High"]), float(row["Low"]), float(row.get("Open", np.nan)), float(row["Close"])
    rng = max(h - l, 1e-12)
    if direction == "long":
        opp = min(o, c) - l
    else:
        opp = h - max(o, c)
    return float(max(opp, 0.0) / rng)

def true_range(high, low, prev_close):
    return np.maximum.reduce([
        (high - low).values,
        np.abs(high - prev_close).values,
        np.abs(low - prev_close).values
    ])

def _apply_slippage(price: float, bps: float, side: str) -> float:
    factor = 1 + (bps/10000.0)
    return price * factor if side == "buy" else price / factor

def _normalize_splits(num_targets: int, splits: tuple) -> List[float]:
    if num_targets <= 0:
        return [1.0]
    if not splits:
        return [1.0 / num_targets] * num_targets
    arr = list(splits[:num_targets])
    if len(arr) < num_targets:
        arr += [0.0] * (num_targets - len(arr))
    s = sum(arr)
    if s <= 0:
        return [1.0 / num_targets] * num_targets
    return [x / s for x in arr]

def print_overall_totals(trades: pd.DataFrame):
    print("\n=== Overall Totals ===")
    if trades is None or trades.empty:
        print("Total trades: 0")
        print("Total PnL ($): 0.00")
        print("Win rate: 0.00%")
        print("Avg R: 0.000 | Median R: 0.000")
        print("Profit Factor: 0.000")
        print("Avg Win ($): 0.00 | Avg Loss ($): 0.00")
        return
    pnl = pd.to_numeric(trades["PnL_$"], errors="coerce").fillna(0.0)
    wins_mask = pnl > 0
    losses_mask = pnl < 0
    total_trades = int(len(pnl))
    total_pnl    = float(pnl.sum())
    win_rate     = float(wins_mask.mean() * 100.0) if total_trades else 0.0
    avg_r        = float(pd.to_numeric(trades["R_multiple"], errors="coerce").mean())
    median_r     = float(pd.to_numeric(trades["R_multiple"], errors="coerce").median())
    gross_profit = float(pnl[wins_mask].sum())
    gross_loss   = float(-pnl[losses_mask].sum())
    pf = (gross_profit / gross_loss) if gross_loss > 0 else (float("inf") if gross_profit > 0 else 0.0)
    avg_win  = float(pnl[wins_mask].mean()) if wins_mask.any() else 0.0
    avg_loss = float(pnl[losses_mask].mean()) if losses_mask.any() else 0.0
    print(f"Total trades: {total_trades}")
    print(f"Total PnL ($): {total_pnl:,.2f}")
    print(f"Win rate: {win_rate:.2f}%")
    print(f"Avg R: {avg_r:.3f} | Median R: {median_r:.3f}")
    print(f"Profit Factor: {pf:.3f}")
    print(f"Avg Win ($): {avg_win:,.2f} | Avg Loss ($): {avg_loss:,.2f}")

# ===== S&P 500 scraping / data =====

WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

def sp500_from_wikipedia() -> pd.DataFrame:
    html = requests.get(WIKI_URL, headers={"User-Agent":"Mozilla/5.0"}, timeout=30).text
    table = pd.read_html(html)[0]
    df = table.rename(columns={"Symbol":"SymbolRaw"}).copy()
    df["Symbol"] = df["SymbolRaw"].astype(str).str.strip()
    df["Ticker"] = df["Symbol"].str.replace(r"\.", "-", regex=True)  # BRK.B -> BRK-B
    keep = ["Ticker","Symbol","Security","GICS Sector","GICS Sub-Industry","Headquarters Location"]
    return df[keep]

def fetch_intraday(
    ticker: str,
    interval: str = INTERVAL,
    tz: str = TZ,
    period: str = PERIOD,
    use_period: bool = USE_PERIOD
) -> pd.DataFrame:
    intraday_small = interval in {"1m","2m","5m","15m","30m"}
    try:
        if intraday_small and use_period:
            df = yf.download(
                ticker, period=period, interval=interval,
                auto_adjust=False, progress=False, group_by="column", threads=True
            )
        else:
            df = yf.download(
                ticker, period=period, interval=interval,
                auto_adjust=False, progress=False, group_by="column", threads=True
            )
    except Exception as e:
        print(f"[{ticker}] download error: {e}")
        return pd.DataFrame()
    if df is None or df.empty:
        print(f"[{ticker}] no data (interval={interval}, period={period})")
        return pd.DataFrame()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] if isinstance(c, tuple) and len(c) else c for c in df.columns]
    if df.index.tz is None:
        df = df.tz_localize("UTC")
    df = df.tz_convert(tz)
    df = df.between_time(REG_SESSION_START, REG_SESSION_END)
    df = df.drop(columns=[c for c in df.columns if str(c).lower().startswith("adj")], errors="ignore")
    df["Ticker"] = ticker
    df = df.sort_index()
    return ensure_unique_columns(df)

# ===== Indicators & features =====

def compute_opening_range(df_15: pd.DataFrame) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    for c in ("ORH","ORL","OR_Open","OR_Close","OR_IBS"):
        if (df.columns == c).sum() > 0:
            df = df.drop(columns=[c])
    for col in ("High","Low","Open","Close"):
        if col not in df.columns:
            raise ValueError(f"compute_opening_range: missing column {col}")
    if not df.index.is_monotonic_increasing:
        df = df.sort_index()
    df["Session"] = sessionize(df)
    first_bar_idx = df.groupby("Session").head(1).index
    df["ORH"] = np.nan; df["ORL"] = np.nan
    df["OR_Open"] = np.nan; df["OR_Close"] = np.nan
    df.loc[first_bar_idx, "ORH"]      = df.loc[first_bar_idx, "High"].astype(float).values
    df.loc[first_bar_idx, "ORL"]      = df.loc[first_bar_idx, "Low"].astype(float).values
    df.loc[first_bar_idx, "OR_Open"]  = df.loc[first_bar_idx, "Open"].astype(float).values
    df.loc[first_bar_idx, "OR_Close"] = df.loc[first_bar_idx, "Close"].astype(float).values
    df[["ORH","ORL","OR_Open","OR_Close"]] = df.groupby("Session")[["ORH","ORL","OR_Open","OR_Close"]].ffill()
    rng = (df["ORH"] - df["ORL"]).replace(0, np.nan)
    df["OR_IBS"] = (df["OR_Close"] - df["ORL"]) / rng
    return ensure_unique_columns(df)

def add_session_vwap(df_15: pd.DataFrame) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df["Session"] = sessionize(df)
    tp = (df["High"] + df["Low"] + df["Close"]) / 3.0
    if "Volume" in df.columns:
        df["vwap_num"] = tp * df["Volume"]
        df["vwap_den"] = df["Volume"].replace(0, np.nan)
    else:
        df["vwap_num"] = tp; df["vwap_den"] = 1.0
    df["VWAP"] = (df.groupby("Session")["vwap_num"].cumsum() /
                  df.groupby("Session")["vwap_den"].cumsum())
    return ensure_unique_columns(df.drop(columns=["vwap_num","vwap_den"]))

def add_ema_trend(df_15: pd.DataFrame, fast=EMA_FAST, slow=EMA_SLOW) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df[f"EMA{fast}"] = df["Close"].ewm(span=fast, adjust=False).mean()
    df[f"EMA{slow}"] = df["Close"].ewm(span=slow, adjust=False).mean()
    return ensure_unique_columns(df)

def rsi(series: pd.Series, length=14) -> pd.Series:
    delta = series.diff()
    up = delta.clip(lower=0); dn = -delta.clip(upper=0)
    avg_gain = up.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = dn.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / (avg_loss.replace(0, np.nan))
    return 100 - (100 / (1 + rs))

def add_rsi(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df[f"RSI{length}"] = rsi(df["Close"], length)
    return ensure_unique_columns(df)

def add_adx(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    up_move = df["High"].diff(); dn_move = -df["Low"].diff()
    plus_dm  = np.where((up_move > dn_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((dn_move > up_move) & (dn_move > 0), dn_move, 0.0)
    prev_close = df["Close"].shift(1)
    tr = pd.Series(true_range(df["High"], df["Low"], prev_close), index=df.index)
    atr = tr.ewm(alpha=1/length, adjust=False).mean()
    plus_di  = 100 * pd.Series(plus_dm, index=df.index).ewm(alpha=1/length, adjust=False).mean() / atr
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1/length, adjust=False).mean() / atr
    dx  = (abs(plus_di - minus_di) / (plus_di + minus_di).replace(0, np.nan)) * 100
    adx = dx.ewm(alpha=1/length, adjust=False).mean()
    df[f"ADX{length}"] = adx
    return ensure_unique_columns(df)

def add_atr_15m(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    prev_close = df["Close"].shift(1)
    tr = pd.Series(true_range(df["High"], df["Low"], prev_close), index=df.index)
    df[f"ATR15_{length}"] = tr.ewm(alpha=1/length, adjust=False).mean()
    return ensure_unique_columns(df)

def build_daily_from_15m(df_15: pd.DataFrame) -> pd.DataFrame:
    d = ensure_unique_columns(df_15.copy())
    d["Session"] = sessionize(d)
    daily = d.groupby("Session").agg(
        Open=("Open","first"),
        High=("High","max"),
        Low=("Low","min"),
        Close=("Close","last"),
        Volume=("Volume","sum") if "Volume" in d.columns else ("Close","size"),
    )
    return daily

def add_daily_bias_and_atr(df_15: pd.DataFrame, ema_fast=20, ema_slow=50, atr_len=14) -> pd.DataFrame:
    dly = build_daily_from_15m(df_15)
    # shift(1) to avoid look-ahead
    dly[f"EMA_D{ema_fast}"] = dly["Close"].ewm(span=ema_fast, adjust=False).mean().shift(1)
    dly[f"EMA_D{ema_slow}"] = dly["Close"].ewm(span=ema_slow, adjust=False).mean().shift(1)
    prev_close = dly["Close"].shift(1)
    tr = pd.Series(np.maximum.reduce([
        (dly["High"] - dly["Low"]).values,
        np.abs(dly["High"] - prev_close).values,
        np.abs(dly["Low"]  - prev_close).values
    ]), index=dly.index)
    atr_col = f"ATR_D{atr_len}"
    dly[atr_col] = tr.ewm(alpha=1/atr_len, adjust=False).mean().shift(1)
    if "Volume" in df_15.columns:
        tmp = df_15.copy(); tmp["Session"] = sessionize(tmp)
        dly["Open15_Vol"] = tmp.groupby("Session")["Volume"].first()
        dly["Open15_Vol_Thresh"] = (
            dly["Open15_Vol"].rolling(window=OPENING_VOL_LOOKBACK, min_periods=10)
            .quantile(OPENING_VOL_QUANTILE).shift(1)
        )
    feature_cols = [f"EMA_D{ema_fast}", f"EMA_D{ema_slow}", atr_col, "Open15_Vol", "Open15_Vol_Thresh"]
    feature_cols = [c for c in feature_cols if c in dly.columns]
    out = ensure_unique_columns(df_15.copy())
    out["Session"] = sessionize(out)
    out = out.merge(dly[feature_cols], left_on="Session", right_index=True, how="left")
    return ensure_unique_columns(out)

def ema_slope_ok(sdf: pd.DataFrame, ts, fast=20, slow=50, min_frac=0.0002):
    window = sdf.loc[:ts].tail(5)
    if f"EMA{fast}" not in window.columns or f"EMA{slow}" not in window.columns:
        return True
    efast = window[f"EMA{fast}"].dropna()
    eslow = window[f"EMA{slow}"].dropna()
    if len(efast) < 2 or len(eslow) < 2:
        return True
    px = float(window["Close"].iloc[-1]); scale = max(px, 1e-6)
    fast_slope = (efast.iloc[-1] - efast.iloc[0]) / scale
    slow_slope = (eslow.iloc[-1] - eslow.iloc[0]) / scale
    return fast_slope, slow_slope

# ===== Backtest (per ticker) =====

def backtest_orb_retest(
    df_15: pd.DataFrame,
    max_retest_min: int = MAX_RETEST_MIN,
    r_targets: List[float] = R_MULTIPLES,
    dollars: float = POSITION_SIZE_DOLLARS,
    slippage_bps: float = SLIPPAGE_BPS,
    fees: float = FEES_PER_TRADE,
    retest_confirm_close: bool = RETEST_CONFIRM_CLOSE,
    spy15: Optional[pd.DataFrame] = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    df = compute_opening_range(df_15)
    if USE_VWAP_FILTER:  df = add_session_vwap(df)
    if USE_TREND_FILTER: df = add_ema_trend(df, EMA_FAST, EMA_SLOW)
    df = add_daily_bias_and_atr(df, DAILY_EMA_FAST, DAILY_EMA_SLOW, DAILY_ATR_LEN)

    df["Session"] = sessionize(df)
    first_idx = df.groupby("Session").head(1).index
    prev_close_series = df.groupby("Session")["Close"].last().shift(1)
    prev_close_map = df["Session"].map(prev_close_series)
    df.loc[first_idx, "GapPct"] = (df.loc[first_idx, "Open"] / prev_close_map.loc[first_idx] - 1.0)
    df["GapPct"] = df.groupby("Session")["GapPct"].ffill()

    if USE_RSI_FILTER:  df = add_rsi(df, RSI_LEN)
    if USE_ADX_FILTER:  df = add_adx(df, ADX_LEN)
    if USE_ATR_TRAIL:   df = add_atr_15m(df, ATR15_LEN)
    df = ensure_unique_columns(df)

    sessions = df["Session"].unique()
    trades = []

    for ses in sessions:
        sdf = df[df["Session"] == ses].copy()
        if len(sdf) < 3: continue

        orh, orl = float(sdf["ORH"].iloc[0]), float(sdf["ORL"].iloc[0])
        if np.isnan(orh) or np.isnan(orl): continue

        after_open = sdf.iloc[1:].copy()
        long_breaks  = after_open[after_open["Close"] > orh * (1 + BREAKOUT_CUSHION_PCT)]
        short_breaks = after_open[after_open["Close"] < orl * (1 - BREAKOUT_CUSHION_PCT)]

        cands = []
        for idx, row in long_breaks.iterrows():  cands.append(("long", idx, row))
        for idx, row in short_breaks.iterrows(): cands.append(("short", idx, row))
        cands.sort(key=lambda x: x[1])

        trades_this_session = 0
        took_long = took_short = False

        for direction, btime, breakout_row in cands:
            if not _clock_le(breakout_row.name, BREAKOUT_DEADLINE): continue
            if not ALLOW_BOTH_SIDES:
                if took_long and direction == "short": break
                if took_short and direction == "long": break
            if direction == "long" and took_long:   continue
            if direction == "short" and took_short: continue

            cutoff = btime + timedelta(minutes=max_retest_min)
            window = sdf[(sdf.index > btime) & (sdf.index <= cutoff)].copy()
            level = orh if direction == "long" else orl

            touch = window[(window["Low"] <= level) & (window["High"] >= level)].head(1)
            retest_used = True
            if touch.empty:
                if USE_CONTINUATION and _clock_le(breakout_row.name, CONTINUATION_CUTOFF):
                    retest_used = False
                    retest_time = breakout_row.name
                    retest_bar = breakout_row
                else:
                    continue
            else:
                retest_bar = touch.iloc[0]
                retest_time = retest_bar.name

            if retest_used and retest_confirm_close:
                if direction == "long":
                    ok = (retest_bar["Close"] >= level * (1 + BREAKOUT_CUSHION_PCT))
                else:
                    ok = (retest_bar["Close"] <= level * (1 - BREAKOUT_CUSHION_PCT))
                if not ok: continue

            # opening bar IBS bias
            ibs = float(sdf["OR_IBS"].iloc[0]) if "OR_IBS" in sdf.columns else 0.5
            if direction == "long" and not (ibs >= 0.55):   continue
            if direction == "short" and not (ibs <= 0.45):  continue

            if MAX_OPPOSITE_WICK_FRAC is not None:
                if wick_opposite_fraction(retest_bar, direction) > MAX_OPPOSITE_WICK_FRAC:
                    continue

            if USE_DOW_FILTER and retest_time.weekday() not in ALLOWED_DOW: continue

            if USE_VWAP_FILTER and "VWAP" in sdf.columns:
                v_prev = sdf.loc[:retest_time, "VWAP"].tail(3).dropna().values
                rising  = (len(v_prev) < 2) or np.all(np.diff(v_prev) >= 0)
                falling = (len(v_prev) < 2) or np.all(np.diff(v_prev) <= 0)
                if direction == "long":
                    if not (retest_bar["Close"] > retest_bar["VWAP"] and rising): continue
                else:
                    if not (retest_bar["Close"] < retest_bar["VWAP"] and falling): continue

            if USE_TREND_FILTER and (f"EMA{EMA_FAST}" in sdf.columns) and (f"EMA{EMA_SLOW}" in sdf.columns):
                emaf = retest_bar.get(f"EMA{EMA_FAST}", np.nan); emas = retest_bar.get(f"EMA{EMA_SLOW}", np.nan)
                if np.isfinite(emaf) and np.isfinite(emas):
                    if direction == "long" and not (emaf > emas): continue
                    if direction == "short" and not (emaf < emas): continue
                slopes = ema_slope_ok(sdf, retest_time, fast=EMA_FAST, slow=EMA_SLOW, min_frac=MIN_EMA_SLOPE_FRAC)
                if slopes is not True:
                    fast_slope, slow_slope = slopes
                    if direction == "long" and not (fast_slope > MIN_EMA_SLOPE_FRAC and slow_slope > 0): continue
                    if direction == "short" and not (fast_slope < -MIN_EMA_SLOPE_FRAC and slow_slope < 0): continue

            if USE_DAILY_TREND_FILTER and f"EMA_D{DAILY_EMA_FAST}" in sdf.columns and f"EMA_D{DAILY_EMA_SLOW}" in sdf.columns:
                dfast = float(sdf[f"EMA_D{DAILY_EMA_FAST}"].iloc[0])
                dslow = float(sdf[f"EMA_D{DAILY_EMA_SLOW}"].iloc[0])
                if direction == "long" and not (dfast > dslow): continue
                if direction == "short" and not (dfast < dslow): continue

            atr_col = f"ATR_D{DAILY_ATR_LEN}"
            if USE_OR_WIDTH_ATR_FILTER and atr_col in sdf.columns:
                or_width = float(orh - orl)
                atr_d = float(sdf[atr_col].iloc[0])
                if atr_d <= 0: continue
                frac = or_width / atr_d
                if not (OR_ATR_MIN <= frac <= OR_ATR_MAX): continue

            if USE_RSI_FILTER and f"RSI{RSI_LEN}" in sdf.columns:
                rsi_val = float(retest_bar[f"RSI{RSI_LEN}"])
                if direction == "long" and not (rsi_val >= RSI_THRESH_LONG): continue
                if direction == "short" and not (rsi_val <= 100 - RSI_THRESH_SHORT): continue

            if USE_ADX_FILTER and f"ADX{ADX_LEN}" in sdf.columns:
                adx_val = float(retest_bar[f"ADX{ADX_LEN}"])
                if not (adx_val >= ADX_MIN): continue

            if USE_GAP_FILTER and "GapPct" in sdf.columns:
                gap = sdf.loc[sdf.index.min(), "GapPct"]
                if pd.notna(gap) and not (GAP_MIN <= abs(float(gap)) <= GAP_MAX): continue

            if ALIGN_WITH_GAP_DIR and "GapPct" in sdf.columns:
                g = float(sdf["GapPct"].iloc[0]) if pd.notna(sdf["GapPct"].iloc[0]) else 0.0
                if (direction == "long" and g < 0) or (direction == "short" and g > 0): continue

            if USE_SPY_CONFIRM and spy15 is not None and retest_time in spy15.index:
                spy_row = spy15.loc[retest_time]
                if direction == "long" and not (spy_row["SPY_Close"] > spy_row["SPY_VWAP"]): continue
                if direction == "short" and not (spy_row["SPY_Close"] < spy_row["SPY_VWAP"]): continue

            if retest_used and (not _clock_le(retest_time, RETEST_DEADLINE)): continue
            if (not retest_used) and (not _clock_le(retest_time, CONTINUATION_CUTOFF)): continue

            entry = _apply_slippage(level, slippage_bps, "buy" if direction == "long" else "sell")
            stop = orl if direction == "long" else orh
            rps = abs(entry - stop)
            if rps <= 1e-12: continue

            qty = POSITION_SIZE_DOLLARS / entry
            side_mult = 1 if direction == "long" else -1

            if USE_SCALE_OUT:
                targets_r = TARGETS_R
            else:
                targets_r = r_targets
            targets = [(entry + r * rps) if direction == "long" else (entry - r * rps) for r in targets_r]

            run = sdf[sdf.index >= retest_time].copy()
            qty_left = qty
            exits = []
            be_armed = (MOVE_STOP_TO_BE_AT_R is not None)
            be_level = entry
            next_tp_idx = 0

            splits = _normalize_splits(len(targets), SCALE_SPLIT) if USE_SCALE_OUT and len(targets) >= 1 else [1.0]
            leg_sizes = [qty * s for s in splits]

            for ts, row in run.iterrows():
                hi, lo = row["High"], row["Low"]

                if be_armed and MOVE_STOP_TO_BE_AT_R is not None:
                    if direction == "long" and hi >= entry + (MOVE_STOP_TO_BE_AT_R * rps):
                        stop = max(stop, be_level); be_armed = False
                    elif direction == "short" and lo <= entry - (MOVE_STOP_TO_BE_AT_R * rps):
                        stop = min(stop, be_level); be_armed = False

                # Trail only after TP2 (index >= 2 means TP3 remainder)
                if USE_ATR_TRAIL and f"ATR15_{ATR15_LEN}" in sdf.columns and qty_left > 1e-9 and next_tp_idx >= 2:
                    atr_now = float(row[f"ATR15_{ATR15_LEN}"])
                    if direction == "long":
                        stop = max(stop, hi - ATR15_MULT * atr_now)
                    else:
                        stop = min(stop, lo + ATR15_MULT * atr_now)

                if lo <= stop <= hi:
                    px = _apply_slippage(stop, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                    label = "breakeven" if abs(px - be_level) < 1e-10 else "stop"
                    exits.append((label, ts, px, qty_left))
                    qty_left = 0; break

                if next_tp_idx < len(targets):
                    tp = targets[next_tp_idx]
                    if lo <= tp <= hi:
                        px = _apply_slippage(tp, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                        fill_qty = leg_sizes[next_tp_idx] if next_tp_idx < len(leg_sizes) else qty_left
                        exits.append((f"tp{next_tp_idx+1}", ts, px, fill_qty))
                        qty_left -= fill_qty
                        next_tp_idx += 1
                        if next_tp_idx == 1 and MOVE_STOP_TO_BE_AT_R is not None:
                            stop = be_level
                        if qty_left <= 1e-9:
                            break

                if USE_CONTINUATION and not _clock_le(ts, CONTINUATION_CUTOFF) and next_tp_idx == 0:
                    stop = be_level

            if qty_left > 1e-9:
                last = run.iloc[-1]
                px = _apply_slippage(last["Close"], SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                exits.append(("eod", run.index[-1], px, qty_left))
                qty_left = 0

            cash_pnl = sum((px - entry) * side_mult * q for (_, _, px, q) in exits) - fees

            trades.append({
                "Ticker": sdf["Ticker"].iloc[0] if "Ticker" in sdf.columns else "",
                "Session": ses,
                "Direction": direction,
                "ORH": orh, "ORL": orl,
                "EntryTime": retest_time,
                "Entry": entry, "Stop": stop,
                "Targets": targets,
                "Exits": [(lab, ts, px, q) for (lab, ts, px, q) in exits],
                "Qty": qty,
                "PnL_$": cash_pnl,
                "R_multiple": cash_pnl / (rps * max(qty, 1e-12))
            })

            if direction == "long":  took_long  = True
            if direction == "short": took_short = True
            trades_this_session += 1
            if trades_this_session >= MAX_TRADES_PER_SESSION: break

    trade_log = pd.DataFrame(trades)
    if trade_log.empty:
        eq = pd.DataFrame(columns=["Session","CumPnL_$"])
        return trade_log, eq
    equity = (trade_log.groupby("Session")["PnL_$"].sum()
              .sort_index().cumsum().reset_index().rename(columns={"PnL_$":"CumPnL_$"}))
    return trade_log, equity

# ===== Optional SPY confirm data =====

def build_spy_confirm() -> Optional[pd.DataFrame]:
    raw = fetch_intraday(SPY_TICKER, interval=INTERVAL, tz=TZ, period=PERIOD, use_period=USE_PERIOD)
    if raw.empty: return None
    spy_df = compute_opening_range(raw.copy())
    spy_df = add_session_vwap(spy_df)
    return spy_df.rename(columns={"VWAP":"SPY_VWAP","Close":"SPY_Close"})[["SPY_VWAP","SPY_Close"]]

# ===== Universe processor =====

def process_universe(tickers: List[str], slippage_bps: float, fees: float, spy15: Optional[pd.DataFrame] = None,
                     autosave_every: int = AUTOSAVE_EVERY, sleep_between: float = SLEEP_BETWEEN_TICKERS) -> Tuple[pd.DataFrame, pd.DataFrame]:
    all_trades, all_equity = [], []
    processed = 0
    for tk in tickers:
        print(f"\n== {tk} ==")
        raw = fetch_intraday(tk, interval=INTERVAL, tz=TZ, period=PERIOD, use_period=USE_PERIOD)
        if raw.empty:
            print("No data."); time.sleep(sleep_between); continue
        df15 = ensure_unique_columns(raw.between_time(REG_SESSION_START, REG_SESSION_END))
        tlog, eq = backtest_orb_retest(
            df15,
            max_retest_min=MAX_RETEST_MIN,
            r_targets=R_MULTIPLES,
            dollars=POSITION_SIZE_DOLLARS,
            slippage_bps=slippage_bps,
            fees=fees,
            retest_confirm_close=RETEST_CONFIRM_CLOSE,
            spy15=spy15
        )
        if not tlog.empty:
            tlog_out = tlog.copy()
            tlog_out["Targets"] = tlog_out["Targets"].apply(lambda xs: ";".join([f"{p:.6f}" for p in xs]))
            tlog_out["Exits"]   = tlog_out["Exits"].apply(lambda xs: ";".join([f"{t[0]}|{t[1]}|{t[2]:.6f}|{t[3]:.6f}" for t in xs]))
            all_trades.append(tlog_out)
        if not eq.empty:
            eq_out = eq.copy(); eq_out["Ticker"] = tk
            all_equity.append(eq_out)
        processed += 1
        if autosave_every and processed % autosave_every == 0:
            if all_trades:
                pd.concat(all_trades, ignore_index=True).to_csv(TRADES_CSV_ALL, index=False)
                print(f"[autosave] wrote {TRADES_CSV_ALL}")
            if all_equity:
                pd.concat(all_equity, ignore_index=True).to_csv(EQUITY_CSV_ALL, index=False)
                print(f"[autosave] wrote {EQUITY_CSV_ALL}")
        time.sleep(sleep_between)
    trades = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    equity = pd.concat(all_equity, ignore_index=True) if all_equity else pd.DataFrame()
    return trades, equity

# =============================
# ========== VALIDATION SUITE ==========
# =============================

def metrics(trades: pd.DataFrame) -> Dict[str, float]:
    """Return a dict with core stats; safe on empty input."""
    if trades is None or trades.empty:
        return dict(trades=0, win_rate=0.0, avg_r=0.0, pf=0.0, pnl=0.0)
    pnl = pd.to_numeric(trades["PnL_$"], errors="coerce").fillna(0.0)
    r   = pd.to_numeric(trades["R_multiple"], errors="coerce").fillna(0.0)
    win = (pnl > 0)
    gp, gl = float(pnl[win].sum()), float(-pnl[~win & (pnl < 0)].sum())
    pf = (gp / gl) if gl > 0 else (float("inf") if gp > 0 else 0.0)
    return dict(
        trades=int(len(trades)),
        win_rate=float(win.mean()*100.0) if len(trades) else 0.0,
        avg_r=float(r.mean()),
        pf=float(pf),
        pnl=float(pnl.sum())
    )

def print_block(title: str, m: Dict[str, float]):
    print(f"\n--- {title} ---")
    print(f"Trades: {m['trades']:,} | Win%: {m['win_rate']:.2f}% | PF: {m['pf']:.3f} | Avg R: {m['avg_r']:.3f} | PnL: ${m['pnl']:,.2f}")

def run_validation_suite():
    # Universe & sector map
    sp = sp500_from_wikipedia()
    tickers = sp["Ticker"].dropna().unique().tolist()
    sectors = sp[["Ticker","GICS Sector"]].rename(columns={"GICS Sector":"Sector"})

    # SPY confirm (if enabled)
    spy15 = build_spy_confirm() if USE_SPY_CONFIRM else None

    # ===== Baseline run (once) with current slippage/fees
    trades_all, equity_all = process_universe(tickers, SLIPPAGE_BPS, FEES_PER_TRADE, spy15=spy15,
                                              autosave_every=0, sleep_between=0.0)

    # Save full universe files
    if not trades_all.empty:
        trades_all.to_csv(TRADES_CSV_ALL, index=False)
    if not equity_all.empty:
        equity_all.to_csv(EQUITY_CSV_ALL, index=False)

    # Baseline metrics
    base = metrics(trades_all)
    print_block("Baseline (all data, current fees/slippage)", base)

    if trades_all.empty:
        print("\nNo trades available; cannot run validations.")
        return

    wr_list = [base["win_rate"]]

    # Helper splits by date
    trades_all["Session_dt"] = pd.to_datetime(trades_all["Session"])
    sessions_sorted = sorted(trades_all["Session_dt"].unique().tolist())
    if len(sessions_sorted) < 20:
        print("\n[Skip] Not enough days for time-based validations.")
    else:
        # 1) Train/Test split (75%/25% by date)
        split_idx = int(len(sessions_sorted) * 0.75)
        split_date = sessions_sorted[split_idx]
        train = trades_all[trades_all["Session_dt"] <= split_date]
        test  = trades_all[trades_all["Session_dt"] >  split_date]
        m_train, m_test = metrics(train), metrics(test)
        print_block(f"Train/Test split (<= {split_date.date()} vs >)", m_test)
        wr_list.append(m_test["win_rate"])

        # 2) Walk-forward (expanding train, next 10 sessions as test)
        wf_winrates = []
        window = max(8, len(sessions_sorted)//12)  # ~12 folds
        for i in range(window, len(sessions_sorted)-1, window):
            tr_dates = sessions_sorted[:i]
            te_dates = sessions_sorted[i:min(i+window, len(sessions_sorted))]
            tr = trades_all[trades_all["Session_dt"].isin(tr_dates)]
            te = trades_all[trades_all["Session_dt"].isin(te_dates)]
            if te.empty or tr.empty:
                continue
            m_te = metrics(te)
            wf_winrates.append(m_te["win_rate"])
        if wf_winrates:
            wf_avg = float(np.mean(wf_winrates))
            print(f"\n--- Walk-Forward Validation ---\nFolds: {len(wf_winrates)} | Avg Test Win%: {wf_avg:.2f}%")
            wr_list.append(wf_avg)
        else:
            print("\n[Skip] Walk-Forward (not enough folds)")

        # 3) Blocked K-fold (time-ordered, purged; train=all before fold, test=that fold)
        K = 5
        fold_size = len(sessions_sorted)//K
        kf_wr = []
        for k in range(1, K):  # start at 1 so there is some training history
            start = k*fold_size
            end   = (k+1)*fold_size if k < K-1 else len(sessions_sorted)
            te_dates = sessions_sorted[start:end]
            tr_dates = sessions_sorted[:start]
            tr = trades_all[trades_all["Session_dt"].isin(tr_dates)]
            te = trades_all[trades_all["Session_dt"].isin(te_dates)]
            if te.empty or tr.empty: continue
            m_te = metrics(te)
            kf_wr.append(m_te["win_rate"])
        if kf_wr:
            kf_avg = float(np.mean(kf_wr))
            print(f"\n--- Blocked K-Fold (time-ordered) ---\nFolds: {len(kf_wr)} | Avg Test Win%: {kf_avg:.2f}%")
            wr_list.append(kf_avg)
        else:
            print("\n[Skip] Blocked K-Fold")

    # 4) Sector holdout tests (out-of-sample instruments)
    trades_sector = trades_all.merge(sectors, on="Ticker", how="left")
    if "Sector" in trades_sector.columns and trades_sector["Sector"].notna().any():
        top_sectors = trades_sector["Sector"].value_counts().head(4).index.tolist()
        holdout_wr = []
        for sec in top_sectors:
            te = trades_sector[trades_sector["Sector"] == sec]
            if te.empty: continue
            m_te = metrics(te)
            print_block(f"Sector Holdout (test={sec})", m_te)
            holdout_wr.append(m_te["win_rate"])
        if holdout_wr:
            wr_list.append(float(np.mean(holdout_wr)))
    else:
        print("\n[Skip] Sector holdout (no sector labels found)")

    # 5) Regime tests (SPY trend/vol regimes from the same 60d universe)
    try:
        spy_raw = fetch_intraday(SPY_TICKER, interval=INTERVAL, tz=TZ, period=PERIOD, use_period=USE_PERIOD)
        if not spy_raw.empty:
            spy_daily = build_daily_from_15m(spy_raw)
            spy_daily["EMA20"] = spy_daily["Close"].ewm(span=20, adjust=False).mean().shift(1)
            spy_daily["EMA50"] = spy_daily["Close"].ewm(span=50, adjust=False).mean().shift(1)
            spy_daily["Trend"] = np.where(spy_daily["EMA20"] > spy_daily["EMA50"], "Bull", "Bear")
            # Vol regime via normalized TR%
            pc = spy_daily["Close"].shift(1)
            trp = np.maximum.reduce([
                (spy_daily["High"]-spy_daily["Low"]).values,
                np.abs(spy_daily["High"]-pc).values,
                np.abs(spy_daily["Low"] -pc).values
            ])/pc.values
            spy_daily["TR_pct"] = trp
            thresh = np.nanmedian(trp)
            spy_daily["VolRegime"] = np.where(spy_daily["TR_pct"] >= thresh, "HighVol", "LowVol")
            td = trades_all.copy()
            td["Session_dt"] = pd.to_datetime(td["Session"])
            td = td.merge(spy_daily[["Trend","VolRegime"]], left_on="Session_dt", right_index=True, how="left")
            for label, subset in [("Bull", td[td["Trend"]=="Bull"]),
                                  ("Bear", td[td["Trend"]=="Bear"]),
                                  ("HighVol", td[td["VolRegime"]=="HighVol"]),
                                  ("LowVol", td[td["VolRegime"]=="LowVol"])]:
                if subset.empty: 
                    print(f"\n[Skip] Regime {label} (no trades)"); 
                    continue
                m = metrics(subset)
                print_block(f"Regime: {label}", m)
                wr_list.append(m["win_rate"])
        else:
            print("\n[Skip] Regime tests (no SPY data)")
    except Exception as e:
        print(f"\n[Skip] Regime tests (error: {e})")

    # 6) Monte-Carlo bootstrap (median WR)
    pnl = pd.to_numeric(trades_all["PnL_$"], errors="coerce").fillna(0.0).values
    if len(pnl) >= 30:
        rng = np.random.default_rng(7)
        trials, m = 1000, []
        N = len(pnl)
        for _ in range(trials):
            sample = pnl[rng.integers(0, N, size=N)]
            m.append((sample > 0).mean()*100.0)
        mc_wr_med = float(np.median(m))
        print(f"\n--- Monte Carlo (bootstrap) ---\nTrials: {trials} | Median Win%: {mc_wr_med:.2f}%")
        wr_list.append(mc_wr_med)
    else:
        print("\n[Skip] Monte Carlo (not enough trades)")

    # 7) Cost sensitivity (fees/slippage)
    sens = []
    for bps in [0.5, 1.0, 2.0]:
        for fee in [0.00, 0.25]:
            # Reprice trades by adjusting PnL for deltas vs current settings
            # (Safer approach is full re-run; here we do a quick re-run on a small subset of tickers)
            # For accuracy, do a small full re-run on the top 50 tickers by count:
            top50 = trades_all["Ticker"].value_counts().head(50).index.tolist()
            spy15_small = build_spy_confirm() if USE_SPY_CONFIRM else None
            tr_small, _ = process_universe(top50, bps, fee, spy15=spy15_small, autosave_every=0, sleep_between=0.0)
            m = metrics(tr_small)
            print_block(f"Cost Sensitivity (slip={bps}bps, fee=${fee:.2f})", m)
            sens.append(m["win_rate"])
    if sens:
        wr_list.append(float(np.mean(sens)))

    # ===== Overall average of Win% across performed tests =====
    wr_used = [w for w in wr_list if np.isfinite(w)]
    if wr_used:
        avg_wr_all_tests = float(np.mean(wr_used))
        print(f"\n=== Average Win% across performed validations: {avg_wr_all_tests:.2f}% ===")
    else:
        print("\nNo Win% values collected from validations.")

    # ===== Final overall stats on full run =====
    print_overall_totals(trades_all)

# ===== Entry point =====
if __name__ == "__main__":
    run_validation_suite()



== MMM ==

== AOS ==

== ABT ==

== ABBV ==

== ACN ==

== ADBE ==

== AMD ==

== AES ==

== AFL ==

== A ==

== APD ==

== ABNB ==

== AKAM ==

== ALB ==

== ARE ==

== ALGN ==

== ALLE ==

== LNT ==

== ALL ==

== GOOGL ==

== GOOG ==

== MO ==

== AMZN ==

== AMCR ==

== AEE ==

== AEP ==

== AXP ==

== AIG ==

== AMT ==

== AWK ==

== AMP ==

== AME ==

== AMGN ==

== APH ==

== ADI ==

== AON ==

== APA ==

== APO ==

== AAPL ==

== AMAT ==

== APTV ==

== ACGL ==

== ADM ==

== ANET ==

== AJG ==

== AIZ ==

== T ==

== ATO ==

== ADSK ==

== ADP ==

== AZO ==

== AVB ==

== AVY ==

== AXON ==

== BKR ==

== BALL ==

== BAC ==

== BAX ==

== BDX ==

== BRK-B ==

== BBY ==

== TECH ==

== BIIB ==

== BLK ==

== BX ==

== XYZ ==

== BK ==

== BA ==

== BKNG ==

== BSX ==

== BMY ==

== AVGO ==

== BR ==

== BRO ==

== BF-B ==

== BLDR ==

== BG ==

== BXP ==

== CHRW ==

== CDNS ==

== CZR ==

== CPT ==

== CPB ==

== COF ==

== CAH ==

== KMX ==

== CCL ==

== CARR ==

== CAT ==


TypeError: Invalid comparison between dtype=datetime64[ns] and int

In [3]:
# orb_15min_retest_sp500_robust.py
# S&P 500, last ~60 trading days (Yahoo 15m).
# ORB (Opening Range Breakout) with Retest/Continuation entries.
# Scale-out in thirds: TP1=1R, TP2=2R, TP3=3R (1/3 each), BE after +1R; optional ATR trail on final third post-TP2.
# Includes: Profit Factor in totals, Monte Carlo helper, slippage/fee sensitivity.
# Anti-overfit harness: train/test split, walk-forward on train to pick a simple config; then test & report.

import warnings
warnings.filterwarnings("ignore")

import time
from datetime import timedelta, time as dtime
from typing import List, Tuple, Optional, Dict
from contextlib import contextmanager

import numpy as np
import pandas as pd
import requests
import yfinance as yf

# =============================
# USER PARAMETERS (defaults; may be overridden by config search)
# =============================

# Data (Yahoo intraday <60m => last ~60d only)
INTERVAL = "15m"
PERIOD = "60d"
USE_PERIOD = True

# Session / timezone
TZ = "America/New_York"
REG_SESSION_START = "09:30"
REG_SESSION_END   = "16:00"

# ORB logic (quality + enough trades)
MAX_RETEST_MIN = 120
RETEST_CONFIRM_CLOSE = True
BREAKOUT_CUSHION_PCT = 0.0012    # 0.12% cushion beyond OR to confirm breakout/retest
MAX_OPPOSITE_WICK_FRAC = 0.30    # reject entry bar if opposite wick > 30% of bar range

# Multiple trades per day
MAX_TRADES_PER_SESSION = 2
ALLOW_BOTH_SIDES = True

# Scale-out in thirds (boost avg R while keeping WR)
USE_SCALE_OUT = True
TARGETS_R = [1.0, 2.0, 3.0]      # 1R, 2R, 3R
SCALE_SPLIT = (1/3, 1/3, 1/3)
MOVE_STOP_TO_BE_AT_R = 1.0       # BE after TP1 (+1R)

# Continuation fallback if no retest
USE_CONTINUATION = True

# Time windows (earlier to avoid chop)
BREAKOUT_DEADLINE   = "10:45"
RETEST_DEADLINE     = "11:30"
CONTINUATION_CUTOFF = "11:30"

# Filters (balanced)
USE_VWAP_FILTER   = True
USE_TREND_FILTER  = True          # 15m EMA trend filter
EMA_FAST = 20
EMA_SLOW = 50
# EMA slope helper (bps of price across last 5 bars)
MIN_EMA_SLOPE_FRAC = 0.0002       # ≈ 2 bps

USE_RSI_FILTER = True
RSI_LEN = 14
RSI_THRESH_LONG = 55.0
RSI_THRESH_SHORT = 45.0

USE_ADX_FILTER = True
ADX_LEN = 14
ADX_MIN = 20.0

USE_DAILY_TREND_FILTER = True
DAILY_EMA_FAST = 20
DAILY_EMA_SLOW = 50
DAILY_ATR_LEN  = 14

# OR width vs daily ATR band
USE_OR_WIDTH_ATR_FILTER = True
OR_ATR_MIN = 0.15
OR_ATR_MAX = 1.20

# Opening 15m volume quantile gate
USE_OPENING_VOL_FILTER = True
OPENING_VOL_LOOKBACK = 60
OPENING_VOL_QUANTILE = 0.40

# Gap filters
USE_GAP_FILTER = True
GAP_MIN = 0.003                 # 0.3%
GAP_MAX = 0.04                  # 4.0%
ALIGN_WITH_GAP_DIR = False

USE_DOW_FILTER = False
ALLOWED_DOW = {1, 2, 3}         # Tue–Thu

# Optional index confirmation
USE_SPY_CONFIRM = True
SPY_TICKER = "SPY"

# Risk / costs
R_MULTIPLES = [2.0]             # unused when USE_SCALE_OUT=True
POSITION_SIZE_DOLLARS = 10_000
SLIPPAGE_BPS = 1.0
FEES_PER_TRADE = 0.00

# Trailing on last third after TP2
USE_ATR_TRAIL = True
ATR15_LEN = 14
ATR15_MULT = 2.5

# Batch / persistence
SLEEP_BETWEEN_TICKERS = 0.35
AUTOSAVE_EVERY = 0  # off in validation harness to avoid extra writes

# Output files
TRADES_CSV_ALL = "orb_trades_sp500.csv"
EQUITY_CSV_ALL = "orb_equity_sp500.csv"
SUMMARY_CSV    = "orb_summary_sp500.csv"

# =============================
# ANTI-OVERFIT HARNESS CONTROLS
# =============================

# For parameter search we only use a subset of tickers for speed
SEARCH_TICKERS_MAX = 60
# Fraction of sessions for training (time-ordered split)
TRAIN_FRACTION = 0.75
# Penalty for complexity: subtract 0.5% WR per enabled “extra” filter
PARSIMONY_PENALTY_PER_FILTER = 0.5  # percentage points
# Within-top band for picking simplest config
WITHIN_TOP_WINRATE_PCT = 1.0  # within 1% WinRate of the best

# =============================
# Helpers / core strategy bits
# =============================

def ensure_unique_columns(df: pd.DataFrame) -> pd.DataFrame:
    if getattr(df.columns, "duplicated", None) is not None and df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()].copy()
    return df

def sessionize(obj) -> pd.Series:
    idx = getattr(obj, "index", obj)
    return pd.to_datetime(pd.Index(idx).date)

def _to_clock(ts) -> dtime:
    return dtime(ts.hour, ts.minute)

def _clock_le(ts, hhmm: str) -> bool:
    h, m = map(int, hhmm.split(":"))
    return _to_clock(ts) <= dtime(h, m)

def wick_opposite_fraction(row: pd.Series, direction: str) -> float:
    h, l, o, c = float(row["High"]), float(row["Low"]), float(row.get("Open", np.nan)), float(row["Close"])
    rng = max(h - l, 1e-12)
    if direction == "long":
        opp = min(o, c) - l
    else:
        opp = h - max(o, c)
    return float(max(opp, 0.0) / rng)

def true_range(high, low, prev_close):
    return np.maximum.reduce([
        (high - low).values,
        np.abs(high - prev_close).values,
        np.abs(low - prev_close).values
    ])

def _apply_slippage(price: float, bps: float, side: str) -> float:
    factor = 1 + (bps/10000.0)
    return price * factor if side == "buy" else price / factor

def _normalize_splits(num_targets: int, splits: tuple) -> List[float]:
    if num_targets <= 0:
        return [1.0]
    if not splits:
        return [1.0 / num_targets] * num_targets
    arr = list(splits[:num_targets])
    if len(arr) < num_targets:
        arr += [0.0] * (num_targets - len(arr))
    s = sum(arr)
    if s <= 0:
        return [1.0 / num_targets] * num_targets
    return [x / s for x in arr]

def print_overall_totals(trades: pd.DataFrame):
    print("\n=== Overall Totals ===")
    if trades is None or trades.empty:
        print("Total trades: 0")
        print("Total PnL ($): 0.00")
        print("Win rate: 0.00%")
        print("Avg R: 0.000 | Median R: 0.000")
        print("Profit Factor: 0.000")
        print("Avg Win ($): 0.00 | Avg Loss ($): 0.00")
        return
    pnl = pd.to_numeric(trades["PnL_$"], errors="coerce").fillna(0.0)
    wins_mask = pnl > 0
    losses_mask = pnl < 0
    total_trades = int(len(pnl))
    total_pnl    = float(pnl.sum())
    win_rate     = float(wins_mask.mean() * 100.0) if total_trades else 0.0
    avg_r        = float(pd.to_numeric(trades["R_multiple"], errors="coerce").mean())
    median_r     = float(pd.to_numeric(trades["R_multiple"], errors="coerce").median())
    gross_profit = float(pnl[wins_mask].sum())
    gross_loss   = float(-pnl[losses_mask].sum())
    pf = (gross_profit / gross_loss) if gross_loss > 0 else (float("inf") if gross_profit > 0 else 0.0)
    avg_win  = float(pnl[wins_mask].mean()) if wins_mask.any() else 0.0
    avg_loss = float(pnl[losses_mask].mean()) if losses_mask.any() else 0.0
    print(f"Total trades: {total_trades}")
    print(f"Total PnL ($): {total_pnl:,.2f}")
    print(f"Win rate: {win_rate:.2f}%")
    print(f"Avg R: {avg_r:.3f} | Median R: {median_r:.3f}")
    print(f"Profit Factor: {pf:.3f}")
    print(f"Avg Win ($): {avg_win:,.2f} | Avg Loss ($): {avg_loss:,.2f}")

# ===== S&P 500 scraping / data =====

WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

def sp500_from_wikipedia() -> pd.DataFrame:
    html = requests.get(WIKI_URL, headers={"User-Agent":"Mozilla/5.0"}, timeout=30).text
    table = pd.read_html(html)[0]
    df = table.rename(columns={"Symbol":"SymbolRaw"}).copy()
    df["Symbol"] = df["SymbolRaw"].astype(str).str.strip()
    df["Ticker"] = df["Symbol"].str.replace(r"\.", "-", regex=True)  # BRK.B -> BRK-B
    keep = ["Ticker","Symbol","Security","GICS Sector","GICS Sub-Industry","Headquarters Location"]
    return df[keep]

def fetch_intraday(
    ticker: str,
    interval: str = INTERVAL,
    tz: str = TZ,
    period: str = PERIOD,
    use_period: bool = USE_PERIOD
) -> pd.DataFrame:
    intraday_small = interval in {"1m","2m","5m","15m","30m"}
    try:
        df = yf.download(
            ticker, period=period, interval=interval,
            auto_adjust=False, progress=False, group_by="column", threads=True
        )
    except Exception as e:
        print(f"[{ticker}] download error: {e}")
        return pd.DataFrame()
    if df is None or df.empty:
        print(f"[{ticker}] no data (interval={interval}, period={period})")
        return pd.DataFrame()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] if isinstance(c, tuple) and len(c) else c for c in df.columns]
    if df.index.tz is None:
        df = df.tz_localize("UTC")
    df = df.tz_convert(tz)
    df = df.between_time(REG_SESSION_START, REG_SESSION_END)
    df = df.drop(columns=[c for c in df.columns if str(c).lower().startswith("adj")], errors="ignore")
    df["Ticker"] = ticker
    df = df.sort_index()
    return ensure_unique_columns(df)

# ===== Indicators & features =====

def compute_opening_range(df_15: pd.DataFrame) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    for c in ("ORH","ORL","OR_Open","OR_Close","OR_IBS"):
        if (df.columns == c).sum() > 0:
            df = df.drop(columns=[c])
    for col in ("High","Low","Open","Close"):
        if col not in df.columns:
            raise ValueError(f"compute_opening_range: missing column {col}")
    if not df.index.is_monotonic_increasing:
        df = df.sort_index()
    df["Session"] = sessionize(df)
    first_bar_idx = df.groupby("Session").head(1).index
    df["ORH"] = np.nan; df["ORL"] = np.nan
    df["OR_Open"] = np.nan; df["OR_Close"] = np.nan
    df.loc[first_bar_idx, "ORH"]      = df.loc[first_bar_idx, "High"].astype(float).values
    df.loc[first_bar_idx, "ORL"]      = df.loc[first_bar_idx, "Low"].astype(float).values
    df.loc[first_bar_idx, "OR_Open"]  = df.loc[first_bar_idx, "Open"].astype(float).values
    df.loc[first_bar_idx, "OR_Close"] = df.loc[first_bar_idx, "Close"].astype(float).values
    df[["ORH","ORL","OR_Open","OR_Close"]] = df.groupby("Session")[["ORH","ORL","OR_Open","OR_Close"]].ffill()
    rng = (df["ORH"] - df["ORL"]).replace(0, np.nan)
    df["OR_IBS"] = (df["OR_Close"] - df["ORL"]) / rng
    return ensure_unique_columns(df)

def add_session_vwap(df_15: pd.DataFrame) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df["Session"] = sessionize(df)
    tp = (df["High"] + df["Low"] + df["Close"]) / 3.0
    if "Volume" in df.columns:
        df["vwap_num"] = tp * df["Volume"]
        df["vwap_den"] = df["Volume"].replace(0, np.nan)
    else:
        df["vwap_num"] = tp; df["vwap_den"] = 1.0
    df["VWAP"] = (df.groupby("Session")["vwap_num"].cumsum() /
                  df.groupby("Session")["vwap_den"].cumsum())
    return ensure_unique_columns(df.drop(columns=["vwap_num","vwap_den"]))

def add_ema_trend(df_15: pd.DataFrame, fast=EMA_FAST, slow=EMA_SLOW) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df[f"EMA{fast}"] = df["Close"].ewm(span=fast, adjust=False).mean()
    df[f"EMA{slow}"] = df["Close"].ewm(span=slow, adjust=False).mean()
    return ensure_unique_columns(df)

def rsi(series: pd.Series, length=14) -> pd.Series:
    delta = series.diff()
    up = delta.clip(lower=0); dn = -delta.clip(upper=0)
    avg_gain = up.ewm(alpha=1/length, adjust=False).mean()
    avg_loss = dn.ewm(alpha=1/length, adjust=False).mean()
    rs = avg_gain / (avg_loss.replace(0, np.nan))
    return 100 - (100 / (1 + rs))

def add_rsi(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    df[f"RSI{length}"] = rsi(df["Close"], length)
    return ensure_unique_columns(df)

def add_adx(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    up_move = df["High"].diff(); dn_move = -df["Low"].diff()
    plus_dm  = np.where((up_move > dn_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((dn_move > up_move) & (dn_move > 0), dn_move, 0.0)
    prev_close = df["Close"].shift(1)
    tr = pd.Series(true_range(df["High"], df["Low"], prev_close), index=df.index)
    atr = tr.ewm(alpha=1/length, adjust=False).mean()
    plus_di  = 100 * pd.Series(plus_dm, index=df.index).ewm(alpha=1/length, adjust=False).mean() / atr
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1/length, adjust=False).mean() / atr
    dx  = (abs(plus_di - minus_di) / (plus_di + minus_di).replace(0, np.nan)) * 100
    adx = dx.ewm(alpha=1/length, adjust=False).mean()
    df[f"ADX{length}"] = adx
    return ensure_unique_columns(df)

def add_atr_15m(df_15: pd.DataFrame, length=14) -> pd.DataFrame:
    df = ensure_unique_columns(df_15.copy())
    prev_close = df["Close"].shift(1)
    tr = pd.Series(true_range(df["High"], df["Low"], prev_close), index=df.index)
    df[f"ATR15_{length}"] = tr.ewm(alpha=1/length, adjust=False).mean()
    return ensure_unique_columns(df)

def build_daily_from_15m(df_15: pd.DataFrame) -> pd.DataFrame:
    d = ensure_unique_columns(df_15.copy())
    d["Session"] = sessionize(d)
    daily = d.groupby("Session").agg(
        Open=("Open","first"),
        High=("High","max"),
        Low=("Low","min"),
        Close=("Close","last"),
        Volume=("Volume","sum") if "Volume" in d.columns else ("Close","size"),
    )
    return daily

def add_daily_bias_and_atr(df_15: pd.DataFrame, ema_fast=20, ema_slow=50, atr_len=14) -> pd.DataFrame:
    dly = build_daily_from_15m(df_15)
    # shift(1) to avoid look-ahead
    dly[f"EMA_D{ema_fast}"] = dly["Close"].ewm(span=ema_fast, adjust=False).mean().shift(1)
    dly[f"EMA_D{ema_slow}"] = dly["Close"].ewm(span=ema_slow, adjust=False).mean().shift(1)
    prev_close = dly["Close"].shift(1)
    tr = pd.Series(np.maximum.reduce([
        (dly["High"] - dly["Low"]).values,
        np.abs(dly["High"] - prev_close).values,
        np.abs(dly["Low"]  - prev_close).values
    ]), index=dly.index)
    atr_col = f"ATR_D{atr_len}"
    dly[atr_col] = tr.ewm(alpha=1/atr_len, adjust=False).mean().shift(1)
    if "Volume" in df_15.columns:
        tmp = df_15.copy(); tmp["Session"] = sessionize(tmp)
        dly["Open15_Vol"] = tmp.groupby("Session")["Volume"].first()
        dly["Open15_Vol_Thresh"] = (
            dly["Open15_Vol"]
            .rolling(window=OPENING_VOL_LOOKBACK, min_periods=10)
            .quantile(OPENING_VOL_QUANTILE)
            .shift(1)
        )
    feature_cols = [f"EMA_D{ema_fast}", f"EMA_D{ema_slow}", atr_col, "Open15_Vol", "Open15_Vol_Thresh"]
    feature_cols = [c for c in feature_cols if c in dly.columns]
    out = ensure_unique_columns(df_15.copy())
    out["Session"] = sessionize(out)
    out = out.merge(dly[feature_cols], left_on="Session", right_index=True, how="left")
    return ensure_unique_columns(out)

def ema_slope_ok(sdf: pd.DataFrame, ts, fast=20, slow=50, min_frac=0.0002):
    window = sdf.loc[:ts].tail(5)
    if f"EMA{fast}" not in window.columns or f"EMA{slow}" not in window.columns:
        return True
    efast = window[f"EMA{fast}"].dropna()
    eslow = window[f"EMA{slow}"].dropna()
    if len(efast) < 2 or len(eslow) < 2:
        return True
    px = float(window["Close"].iloc[-1]); scale = max(px, 1e-6)
    fast_slope = (efast.iloc[-1] - efast.iloc[0]) / scale
    slow_slope = (eslow.iloc[-1] - eslow.iloc[0]) / scale
    return fast_slope, slow_slope

# ===== Backtest (per ticker) =====

def backtest_orb_retest(
    df_15: pd.DataFrame,
    max_retest_min: int = MAX_RETEST_MIN,
    r_targets: List[float] = R_MULTIPLES,
    dollars: float = POSITION_SIZE_DOLLARS,
    slippage_bps: float = SLIPPAGE_BPS,
    fees: float = FEES_PER_TRADE,
    retest_confirm_close: bool = RETEST_CONFIRM_CLOSE,
    spy15: Optional[pd.DataFrame] = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    df = compute_opening_range(df_15)
    if USE_VWAP_FILTER:  df = add_session_vwap(df)
    if USE_TREND_FILTER: df = add_ema_trend(df, EMA_FAST, EMA_SLOW)
    df = add_daily_bias_and_atr(df, DAILY_EMA_FAST, DAILY_EMA_SLOW, DAILY_ATR_LEN)

    df["Session"] = sessionize(df)
    first_idx = df.groupby("Session").head(1).index
    prev_close_series = df.groupby("Session")["Close"].last().shift(1)
    prev_close_map = df["Session"].map(prev_close_series)
    df.loc[first_idx, "GapPct"] = (df.loc[first_idx, "Open"] / prev_close_map.loc[first_idx] - 1.0)
    df["GapPct"] = df.groupby("Session")["GapPct"].ffill()

    if USE_RSI_FILTER:  df = add_rsi(df, RSI_LEN)
    if USE_ADX_FILTER:  df = add_adx(df, ADX_LEN)
    if USE_ATR_TRAIL:   df = add_atr_15m(df, ATR15_LEN)
    df = ensure_unique_columns(df)

    sessions = df["Session"].unique()
    trades = []

    for ses in sessions:
        sdf = df[df["Session"] == ses].copy()
        if len(sdf) < 3: continue

        orh, orl = float(sdf["ORH"].iloc[0]), float(sdf["ORL"].iloc[0])
        if np.isnan(orh) or np.isnan(orl): continue

        after_open = sdf.iloc[1:].copy()
        long_breaks  = after_open[after_open["Close"] > orh * (1 + BREAKOUT_CUSHION_PCT)]
        short_breaks = after_open[after_open["Close"] < orl * (1 - BREAKOUT_CUSHION_PCT)]

        cands = []
        for idx, row in long_breaks.iterrows():  cands.append(("long", idx, row))
        for idx, row in short_breaks.iterrows(): cands.append(("short", idx, row))
        cands.sort(key=lambda x: x[1])

        trades_this_session = 0
        took_long = took_short = False

        for direction, btime, breakout_row in cands:
            if not _clock_le(breakout_row.name, BREAKOUT_DEADLINE): continue
            if not ALLOW_BOTH_SIDES:
                if took_long and direction == "short": break
                if took_short and direction == "long": break
            if direction == "long" and took_long:   continue
            if direction == "short" and took_short: continue

            cutoff = btime + timedelta(minutes=max_retest_min)
            window = sdf[(sdf.index > btime) & (sdf.index <= cutoff)].copy()
            level = orh if direction == "long" else orl

            touch = window[(window["Low"] <= level) & (window["High"] >= level)].head(1)
            retest_used = True
            if touch.empty:
                if USE_CONTINUATION and _clock_le(breakout_row.name, CONTINUATION_CUTOFF):
                    retest_used = False
                    retest_time = breakout_row.name
                    retest_bar = breakout_row
                else:
                    continue
            else:
                retest_bar = touch.iloc[0]
                retest_time = retest_bar.name

            if retest_used and retest_confirm_close:
                if direction == "long":
                    ok = (retest_bar["Close"] >= level * (1 + BREAKOUT_CUSHION_PCT))
                else:
                    ok = (retest_bar["Close"] <= level * (1 - BREAKOUT_CUSHION_PCT))
                if not ok: continue

            # Opening bar IBS bias
            ibs = float(sdf["OR_IBS"].iloc[0]) if "OR_IBS" in sdf.columns else 0.5
            if direction == "long" and not (ibs >= 0.55):   continue
            if direction == "short" and not (ibs <= 0.45):  continue

            if MAX_OPPOSITE_WICK_FRAC is not None:
                if wick_opposite_fraction(retest_bar, direction) > MAX_OPPOSITE_WICK_FRAC:
                    continue

            if USE_DOW_FILTER and retest_time.weekday() not in ALLOWED_DOW: continue

            if USE_VWAP_FILTER and "VWAP" in sdf.columns:
                v_prev = sdf.loc[:retest_time, "VWAP"].tail(3).dropna().values
                rising  = (len(v_prev) < 2) or np.all(np.diff(v_prev) >= 0)
                falling = (len(v_prev) < 2) or np.all(np.diff(v_prev) <= 0)
                if direction == "long":
                    if not (retest_bar["Close"] > retest_bar["VWAP"] and rising): continue
                else:
                    if not (retest_bar["Close"] < retest_bar["VWAP"] and falling): continue

            if USE_TREND_FILTER and (f"EMA{EMA_FAST}" in sdf.columns) and (f"EMA{EMA_SLOW}" in sdf.columns):
                emaf = retest_bar.get(f"EMA{EMA_FAST}", np.nan); emas = retest_bar.get(f"EMA{EMA_SLOW}", np.nan)
                if np.isfinite(emaf) and np.isfinite(emas):
                    if direction == "long" and not (emaf > emas): continue
                    if direction == "short" and not (emaf < emas): continue
                slopes = ema_slope_ok(sdf, retest_time, fast=EMA_FAST, slow=EMA_SLOW, min_frac=MIN_EMA_SLOPE_FRAC)
                if slopes is not True:
                    fast_slope, slow_slope = slopes
                    if direction == "long" and not (fast_slope > MIN_EMA_SLOPE_FRAC and slow_slope > 0): continue
                    if direction == "short" and not (fast_slope < -MIN_EMA_SLOPE_FRAC and slow_slope < 0): continue

            if USE_DAILY_TREND_FILTER and f"EMA_D{DAILY_EMA_FAST}" in sdf.columns and f"EMA_D{DAILY_EMA_SLOW}" in sdf.columns:
                dfast = float(sdf[f"EMA_D{DAILY_EMA_FAST}"].iloc[0])
                dslow = float(sdf[f"EMA_D{DAILY_EMA_SLOW}"].iloc[0])
                if direction == "long" and not (dfast > dslow): continue
                if direction == "short" and not (dfast < dslow): continue

            atr_col = f"ATR_D{DAILY_ATR_LEN}"
            if USE_OR_WIDTH_ATR_FILTER and atr_col in sdf.columns:
                or_width = float(orh - orl)
                atr_d = float(sdf[atr_col].iloc[0])
                if atr_d <= 0: continue
                frac = or_width / atr_d
                if not (OR_ATR_MIN <= frac <= OR_ATR_MAX): continue

            if USE_RSI_FILTER and f"RSI{RSI_LEN}" in sdf.columns:
                rsi_val = float(retest_bar[f"RSI{RSI_LEN}"])
                if direction == "long" and not (rsi_val >= RSI_THRESH_LONG): continue
                if direction == "short" and not (rsi_val <= 100 - RSI_THRESH_SHORT): continue

            if USE_ADX_FILTER and f"ADX{ADX_LEN}" in sdf.columns:
                adx_val = float(retest_bar[f"ADX{ADX_LEN}"])
                if not (adx_val >= ADX_MIN): continue

            if USE_GAP_FILTER and "GapPct" in sdf.columns:
                gap = sdf.loc[sdf.index.min(), "GapPct"]
                if pd.notna(gap) and not (GAP_MIN <= abs(float(gap)) <= GAP_MAX): continue

            if ALIGN_WITH_GAP_DIR and "GapPct" in sdf.columns:
                g = float(sdf["GapPct"].iloc[0]) if pd.notna(sdf["GapPct"].iloc[0]) else 0.0
                if (direction == "long" and g < 0) or (direction == "short" and g > 0): continue

            if USE_SPY_CONFIRM and spy15 is not None and retest_time in spy15.index:
                spy_row = spy15.loc[retest_time]
                if direction == "long" and not (spy_row["SPY_Close"] > spy_row["SPY_VWAP"]): continue
                if direction == "short" and not (spy_row["SPY_Close"] < spy_row["SPY_VWAP"]): continue

            if retest_used and (not _clock_le(retest_time, RETEST_DEADLINE)): continue
            if (not retest_used) and (not _clock_le(retest_time, CONTINUATION_CUTOFF)): continue

            entry = _apply_slippage(level, slippage_bps, "buy" if direction == "long" else "sell")
            stop = orl if direction == "long" else orh
            rps = abs(entry - stop)
            if rps <= 1e-12: continue

            qty = POSITION_SIZE_DOLLARS / entry
            side_mult = 1 if direction == "long" else -1

            if USE_SCALE_OUT:
                targets_r = TARGETS_R
            else:
                targets_r = r_targets
            targets = [(entry + r * rps) if direction == "long" else (entry - r * rps) for r in targets_r]

            run = sdf[sdf.index >= retest_time].copy()
            qty_left = qty
            exits = []
            be_armed = (MOVE_STOP_TO_BE_AT_R is not None)
            be_level = entry
            next_tp_idx = 0

            splits = _normalize_splits(len(targets), SCALE_SPLIT) if USE_SCALE_OUT and len(targets) >= 1 else [1.0]
            leg_sizes = [qty * s for s in splits]

            for ts, row in run.iterrows():
                hi, lo = row["High"], row["Low"]

                if be_armed and MOVE_STOP_TO_BE_AT_R is not None:
                    if direction == "long" and hi >= entry + (MOVE_STOP_TO_BE_AT_R * rps):
                        stop = max(stop, be_level); be_armed = False
                    elif direction == "short" and lo <= entry - (MOVE_STOP_TO_BE_AT_R * rps):
                        stop = min(stop, be_level); be_armed = False

                # Trail only the last third AFTER TP2
                if USE_ATR_TRAIL and f"ATR15_{ATR15_LEN}" in sdf.columns and qty_left > 1e-9 and next_tp_idx >= 2:
                    atr_now = float(row[f"ATR15_{ATR15_LEN}"])
                    if direction == "long":
                        stop = max(stop, hi - ATR15_MULT * atr_now)
                    else:
                        stop = min(stop, lo + ATR15_MULT * atr_now)

                if lo <= stop <= hi:
                    px = _apply_slippage(stop, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                    label = "breakeven" if abs(px - be_level) < 1e-10 else "stop"
                    exits.append((label, ts, px, qty_left))
                    qty_left = 0; break

                if next_tp_idx < len(targets):
                    tp = targets[next_tp_idx]
                    if lo <= tp <= hi:
                        px = _apply_slippage(tp, SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                        fill_qty = leg_sizes[next_tp_idx] if next_tp_idx < len(leg_sizes) else qty_left
                        exits.append((f"tp{next_tp_idx+1}", ts, px, fill_qty))
                        qty_left -= fill_qty
                        next_tp_idx += 1
                        if next_tp_idx == 1 and MOVE_STOP_TO_BE_AT_R is not None:
                            stop = be_level
                        if qty_left <= 1e-9:
                            break

                if USE_CONTINUATION and not _clock_le(ts, CONTINUATION_CUTOFF) and next_tp_idx == 0:
                    stop = be_level

            if qty_left > 1e-9:
                last = run.iloc[-1]
                px = _apply_slippage(last["Close"], SLIPPAGE_BPS, "sell" if direction == "long" else "buy")
                exits.append(("eod", run.index[-1], px, qty_left))
                qty_left = 0

            cash_pnl = sum((px - entry) * side_mult * q for (_, _, px, q) in exits) - fees

            trades.append({
                "Ticker": sdf["Ticker"].iloc[0] if "Ticker" in sdf.columns else "",
                "Session": ses,
                "Direction": direction,
                "ORH": orh, "ORL": orl,
                "EntryTime": retest_time,
                "Entry": entry, "Stop": stop,
                "Targets": targets,
                "Exits": [(lab, ts, px, q) for (lab, ts, px, q) in exits],
                "Qty": qty,
                "PnL_$": cash_pnl,
                "R_multiple": cash_pnl / (rps * max(qty, 1e-12))
            })

            if direction == "long":  took_long  = True
            if direction == "short": took_short = True
            trades_this_session += 1
            if trades_this_session >= MAX_TRADES_PER_SESSION: break

    trade_log = pd.DataFrame(trades)
    if trade_log.empty:
        eq = pd.DataFrame(columns=["Session","CumPnL_$"])
        return trade_log, eq
    equity = (trade_log.groupby("Session")["PnL_$"].sum()
              .sort_index().cumsum().reset_index().rename(columns={"PnL_$":"CumPnL_$"}))
    return trade_log, equity

# ===== Optional SPY confirm data =====

def build_spy_confirm() -> Optional[pd.DataFrame]:
    raw = fetch_intraday(SPY_TICKER, interval=INTERVAL, tz=TZ, period=PERIOD, use_period=USE_PERIOD)
    if raw.empty: return None
    spy_df = compute_opening_range(raw.copy())
    spy_df = add_session_vwap(spy_df)
    return spy_df.rename(columns={"VWAP":"SPY_VWAP","Close":"SPY_Close"})[["SPY_VWAP","SPY_Close"]]

# ===== Universe processor =====

def process_universe(tickers: List[str], slippage_bps: float, fees: float, spy15: Optional[pd.DataFrame] = None,
                     autosave_every: int = AUTOSAVE_EVERY, sleep_between: float = SLEEP_BETWEEN_TICKERS) -> Tuple[pd.DataFrame, pd.DataFrame]:
    all_trades, all_equity = [], []
    processed = 0
    for tk in tickers:
        print(f"\n== {tk} ==")
        raw = fetch_intraday(tk, interval=INTERVAL, tz=TZ, period=PERIOD, use_period=USE_PERIOD)
        if raw.empty:
            print("No data."); time.sleep(sleep_between); continue
        df15 = ensure_unique_columns(raw.between_time(REG_SESSION_START, REG_SESSION_END))
        tlog, eq = backtest_orb_retest(
            df15,
            max_retest_min=MAX_RETEST_MIN,
            r_targets=R_MULTIPLES,
            dollars=POSITION_SIZE_DOLLARS,
            slippage_bps=slippage_bps,
            fees=fees,
            retest_confirm_close=RETEST_CONFIRM_CLOSE,
            spy15=spy15
        )
        if not tlog.empty:
            tlog_out = tlog.copy()
            tlog_out["Targets"] = tlog_out["Targets"].apply(lambda xs: ";".join([f"{p:.6f}" for p in xs]))
            tlog_out["Exits"]   = tlog_out["Exits"].apply(lambda xs: ";".join([f"{t[0]}|{t[1]}|{t[2]:.6f}|{t[3]:.6f}" for t in xs]))
            all_trades.append(tlog_out)
        if not eq.empty:
            eq_out = eq.copy(); eq_out["Ticker"] = tk
            all_equity.append(eq_out)
        processed += 1
        if autosave_every and processed % autosave_every == 0:
            if all_trades:
                pd.concat(all_trades, ignore_index=True).to_csv(TRADES_CSV_ALL, index=False)
                print(f"[autosave] wrote {TRADES_CSV_ALL}")
            if all_equity:
                pd.concat(all_equity, ignore_index=True).to_csv(EQUITY_CSV_ALL, index=False)
                print(f"[autosave] wrote {EQUITY_CSV_ALL}")
        time.sleep(sleep_between)
    trades = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    equity = pd.concat(all_equity, ignore_index=True) if all_equity else pd.DataFrame()
    return trades, equity

# =============================
# VALIDATION + ANTI-OVERFIT
# =============================

def metrics(trades: pd.DataFrame) -> Dict[str, float]:
    if trades is None or trades.empty:
        return dict(trades=0, win_rate=0.0, avg_r=0.0, pf=0.0, pnl=0.0)
    pnl = pd.to_numeric(trades["PnL_$"], errors="coerce").fillna(0.0)
    r   = pd.to_numeric(trades["R_multiple"], errors="coerce").fillna(0.0)
    win = (pnl > 0)
    gp, gl = float(pnl[win].sum()), float(-pnl[~win & (pnl < 0)].sum())
    pf = (gp / gl) if gl > 0 else (float("inf") if gp > 0 else 0.0)
    return dict(
        trades=int(len(trades)),
        win_rate=float(win.mean()*100.0) if len(trades) else 0.0,
        avg_r=float(r.mean()),
        pf=float(pf),
        pnl=float(pnl.sum())
    )

def print_block(title: str, m: Dict[str, float]):
    print(f"\n--- {title} ---")
    print(f"Trades: {m['trades']:,} | Win%: {m['win_rate']:.2f}% | PF: {m['pf']:.3f} | Avg R: {m['avg_r']:.3f} | PnL: ${m['pnl']:,.2f}")

@contextmanager
def temporary_overrides(**kwargs):
    """Temporarily override module-level params for a run; restore after."""
    old = {}
    for k, v in kwargs.items():
        old[k] = globals().get(k, None)
        globals()[k] = v
    try:
        yield
    finally:
        for k, v in old.items():
            globals()[k] = v

def walk_forward_avg_wr(trades: pd.DataFrame, folds: int = 6) -> float:
    """Expanding-window walk-forward Win% on the provided trades (time-ordered)."""
    if trades.empty: return 0.0
    t = trades.copy()
    t["Session_dt"] = pd.to_datetime(t["Session"]).dt.tz_localize(None)
    sessions = np.sort(t["Session_dt"].dropna().unique())
    if len(sessions) < (folds + 2):
        return float((pd.to_numeric(t["PnL_$"], errors="coerce").fillna(0.0) > 0).mean() * 100.0)
    window = max(5, len(sessions)//folds)
    wrs = []
    for i in range(window, len(sessions)-1, window):
        tr_dates = sessions[:i]
        te_dates = sessions[i: min(i+window, len(sessions))]
        tr = t[t["Session_dt"].isin(tr_dates)]
        te = t[t["Session_dt"].isin(te_dates)]
        if te.empty or tr.empty: continue
        wrs.append(float((pd.to_numeric(te["PnL_$"], errors="coerce") > 0).mean()*100.0))
    return float(np.mean(wrs)) if wrs else 0.0

def config_complexity(cfg: Dict) -> int:
    """Count of 'extra' filters toggled on."""
    extras = ["USE_RSI_FILTER","USE_ADX_FILTER","USE_SPY_CONFIRM"]
    return sum(int(cfg.get(k, globals().get(k))) for k in extras)

def run_with_config(tickers: List[str], cfg: Dict, slippage_bps: float, fee: float) -> pd.DataFrame:
    spy15 = build_spy_confirm() if (cfg.get("USE_SPY_CONFIRM", USE_SPY_CONFIRM)) else None
    with temporary_overrides(**cfg):
        trades, _ = process_universe(tickers, slippage_bps=slippage_bps, fees=fee, spy15=spy15,
                                     autosave_every=0, sleep_between=0.0)
    return trades

def robust_config_search(all_tickers: List[str], train_dates: np.ndarray) -> Dict:
    """Search a small grid of configs on a subset of tickers using ONLY training dates. Pick simple config near top."""
    subset = all_tickers[:SEARCH_TICKERS_MAX]

    # Small grid (kept tiny for runtime)
    grid = []
    for rsi in [False, True]:
        for adx in [False, True]:
            for spy in [False, True]:
                for cush in [0.0005, 0.0012]:
                    for wick in [0.30, 0.40]:
                        grid.append(dict(
                            USE_RSI_FILTER=rsi,
                            USE_ADX_FILTER=adx,
                            USE_SPY_CONFIRM=spy,
                            BREAKOUT_CUSHION_PCT=cush,
                            MAX_OPPOSITE_WICK_FRAC=wick
                        ))

    results = []
    for cfg in grid:
        tr = run_with_config(subset, cfg, SLIPPAGE_BPS, FEES_PER_TRADE)
        if tr.empty: 
            results.append((cfg, 0.0, 0.0)); 
            continue
        tr["Session_dt"] = pd.to_datetime(tr["Session"]).dt.tz_localize(None)
        tr_train = tr[tr["Session_dt"].isin(train_dates)]
        wf_wr = walk_forward_avg_wr(tr_train, folds=6)
        # penalize complexity
        score = wf_wr - PARSIMONY_PENALTY_PER_FILTER * config_complexity(cfg)
        results.append((cfg, wf_wr, score))

    if not results:
        return {}  # fallback to defaults

    # pick best by score; then pick simplest within WITHIN_TOP_WINRATE_PCT of top wf_wr
    results.sort(key=lambda x: x[2], reverse=True)  # by score
    top_wr = max(wf for (_, wf, _) in results)
    candidates = [(cfg, wf, sc) for (cfg, wf, sc) in results if (top_wr - wf) <= WITHIN_TOP_WINRATE_PCT]
    if not candidates:
        best_cfg = results[0][0]
    else:
        # choose minimal complexity; tie-break by higher score
        candidates.sort(key=lambda x: (config_complexity(x[0]), -x[2]))
        best_cfg = candidates[0][0]

    print("\n=== Chosen Config (anti-overfit) ===")
    for k, v in best_cfg.items():
        print(f"{k}: {v}")
    return best_cfg

def run_validation_suite():
    sp = sp500_from_wikipedia()
    tickers = sp["Ticker"].dropna().unique().tolist()
    print(f"Tickers loaded: {len(tickers)}")

    # ==== Single baseline run on defaults (for reference) ====
    spy15_base = build_spy_confirm() if USE_SPY_CONFIRM else None
    trades_base, _ = process_universe(tickers, SLIPPAGE_BPS, FEES_PER_TRADE, spy15=spy15_base,
                                      autosave_every=0, sleep_between=0.0)
    print_block("Baseline (defaults)", metrics(trades_base))

    if trades_base.empty:
        print("\nNo trades available; cannot run validations.")
        return

    # ==== Time-ordered Train/Test split (FIX for your error) ====
    t = trades_base.copy()
    t["Session_dt"] = pd.to_datetime(t["Session"]).dt.tz_localize(None)  # ensure Timestamp dtype
    sessions_sorted = np.sort(t["Session_dt"].dropna().unique())
    if len(sessions_sorted) < 20:
        print("\n[Skip] Not enough unique sessions for robust validation.")
        print_overall_totals(trades_base)
        return

    split_idx = int(len(sessions_sorted) * TRAIN_FRACTION)
    split_date = pd.Timestamp(sessions_sorted[split_idx])  # ensure Timestamp, not int
    train_dates = sessions_sorted[:split_idx]
    test_dates  = sessions_sorted[split_idx:]

    # ==== Config search on TRAIN only (subset of tickers) ====
    best_cfg = robust_config_search(tickers, train_dates)
    if not best_cfg:
        best_cfg = {}  # fallback (use defaults)

    # ==== Final eval with chosen config on FULL universe ====
    trades_full = run_with_config(tickers, best_cfg, SLIPPAGE_BPS, FEES_PER_TRADE)
    if trades_full.empty:
        print("\n[Chosen config produced no trades on full universe.]")
        return

    trades_full["Session_dt"] = pd.to_datetime(trades_full["Session"]).dt.tz_localize(None)
    train_trades = trades_full[trades_full["Session_dt"].isin(train_dates)]
    test_trades  = trades_full[trades_full["Session_dt"].isin(test_dates)]

    print_block(f"Train ({len(train_dates)} days)", metrics(train_trades))
    print_block(f"Test  ({len(test_dates)} days)",  metrics(test_trades))

    # ==== Overall stats on chosen config ====
    print_block("Overall (chosen config)", metrics(trades_full))
    print_overall_totals(trades_full)

    # Save if you want artifacts
    trades_full.to_csv(TRADES_CSV_ALL, index=False)

# ===== Entry point =====
if __name__ == "__main__":
    run_validation_suite()


Tickers loaded: 503

== MMM ==

== AOS ==

== ABT ==

== ABBV ==

== ACN ==

== ADBE ==

== AMD ==

== AES ==

== AFL ==

== A ==

== APD ==

== ABNB ==

== AKAM ==

== ALB ==

== ARE ==

== ALGN ==

== ALLE ==

== LNT ==

== ALL ==

== GOOGL ==

== GOOG ==

== MO ==

== AMZN ==

== AMCR ==

== AEE ==

== AEP ==

== AXP ==

== AIG ==

== AMT ==

== AWK ==

== AMP ==

== AME ==

== AMGN ==

== APH ==

== ADI ==

== AON ==

== APA ==

== APO ==

== AAPL ==

== AMAT ==

== APTV ==

== ACGL ==

== ADM ==

== ANET ==

== AJG ==

== AIZ ==

== T ==

== ATO ==

== ADSK ==

== ADP ==

== AZO ==

== AVB ==

== AVY ==

== AXON ==

== BKR ==

== BALL ==

== BAC ==

== BAX ==

== BDX ==

== BRK-B ==

== BBY ==

== TECH ==

== BIIB ==

== BLK ==

== BX ==

== XYZ ==

== BK ==

== BA ==

== BKNG ==

== BSX ==

== BMY ==

== AVGO ==

== BR ==

== BRO ==

== BF-B ==

== BLDR ==

== BG ==

== BXP ==

== CHRW ==

== CDNS ==

== CZR ==

== CPT ==

== CPB ==

== COF ==

== CAH ==

== KMX ==

== CCL ==

==